In [1]:
# main2.py

# CISB 63 - Python AI Beginners
# AI Essay Checker built with Streamlit + OpenAI
#
# This app lets the user upload an essay (PDF or TXT),
# sends the text to an AI model, and returns detailed feedback.

!pip install matplotlib-venn
!pip install streamlit
!pip install PyPDF2
!pip install langgraph langchain python-dotenv langchain-openai
!pip install openai

# ---------------------------------------------------------
# 0. IMPORT LIBRARIES
# ---------------------------------------------------------


import streamlit as st           # Web UI framework
from dotenv import load_dotenv   # To load environment variables from .env
from PyPDF2 import PdfReader     # To read text from PDF files
import os                        # To access environment variables
from openai import OpenAI        # Official OpenAI Python client
from google.colab import userdata # To access Colab secrets

# ---------------------------------------------------------
# 1. INITIAL SETUP
# ---------------------------------------------------------

# Load variables from .env file (for example: OPENAI_API_KEY)
# This keeps your secret key out of the code.
load_dotenv()

# Create an OpenAI client using the API key from the environment.
# Make sure your .env file has a line like:
# OPENAI_API_KEY=sk-xxxxxx
# In Colab, you can also store your API key in secrets and access it via userdata.get('OPENAI_API_KEY')
api_key = os.getenv("OPENAI_API_KEY") or userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

# ---------------------------------------------------------
# 2. HELPER FUNCTIONS
# ---------------------------------------------------------

def extract_text_from_pdf(pdf_file):
    """
    Read every page of a PDF file and return all the extracted text.

    Parameters:
        pdf_file: file-like object uploaded through Streamlit.

    Returns:
        A single string containing the text of the entire PDF.
    """
    reader = PdfReader(pdf_file)     # Create a PDF reader object
    text_chunks = []                 # We will collect text from each page here

    # Loop through each page in the PDF and extract text
    for page in reader.pages:
        page_text = page.extract_text() or ""  # Some pages may return None
        text_chunks.append(page_text)

    # Join all pages together with new lines
    return "\n".join(text_chunks)


def extract_text_from_uploaded_file(uploaded_file):
    """
    Decide how to read the uploaded file based on its MIME type.

    We support:
      - PDF files
      - Plain text (.txt) files

    Parameters:
        uploaded_file: file uploaded via Streamlit.

    Returns:
        The text content as a string, or None if the type is unsupported.
    """
    if uploaded_file is None:
        return None

    # 'application/pdf' is the MIME type Streamlit uses for PDFs
    if uploaded_file.type == "application/pdf":
        return extract_text_from_pdf(uploaded_file)

    # 'text/plain' is the MIME type for standard .txt files
    elif uploaded_file.type == "text/plain":
        # Read the bytes and decode to a normal Python string (UTF-8)
        bytes_data = uploaded_file.read()
        return bytes_data.decode("utf-8", errors="ignore")

    # Any other file type will not be processed
    else:
        return None


def get_essay_feedback(essay_text, assignment_details=None):
    """
    Send the essay text (and optional assignment / rubric details)
    to the OpenAI Chat Completions API and return the AI's feedback.

    Parameters:
        essay_text: the full essay as a string.
        assignment_details: optional description of the prompt or rubric.

    Returns:
        A string with detailed feedback generated by the AI model.
    """

    # Safety check: make sure we actually received some text
    if not essay_text.strip():
        return "I could not detect any text in the uploaded file."

    # System message tells the model how it should behave.
    system_message = (
        "You are a helpful writing tutor. "
        "You carefully read student essays and give specific, constructive feedback. "
        "Focus on structure, clarity, grammar, vocabulary, and whether the essay answers the prompt."
    )

    # Build the user message differently depending on whether
    # the user provided assignment/rubric information.
    if assignment_details:
        user_prompt = f"""
Here is an essay a student wrote.

Assignment / prompt or rubric:
{assignment_details}

Essay:
{essay_text}

Please:
1. Briefly summarize what the essay is trying to say.
2. Comment on strengths (organization, ideas, evidence, style).
3. Point out issues (clarity, grammar, word choice, missing parts).
4. Suggest concrete improvements and examples of better sentences.
5. (Optional) Give a rough letter grade (A–F) or score out of 100 and explain why.
"""
    else:
        user_prompt = f"""
Here is a student's essay:

{essay_text}

Please:
1. Briefly summarize what the essay is trying to say.
2. Comment on strengths (organization, ideas, evidence, style).
3. Point out issues (clarity, grammar, word choice, missing parts).
4. Suggest concrete improvements and examples of better sentences.
5. (Optional) Give a rough letter grade (A–F) or score out of 100 and explain why.
"""

    # Call the OpenAI Chat Completions API.
    # You can change the model name if your instructor wants a different one.
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,   # Low temperature = more focused / consistent feedback
    )

    # Extract the assistant's message text from the API response
    return response.choices[0].message.content

# ---------------------------------------------------------
# 3. STREAMLIT USER INTERFACE
# ---------------------------------------------------------

# Configure the Streamlit page
st.set_page_config(
    page_title="AI Essay Checker",
    page_icon="📝",
    layout="centered"
)

# Main title and description
st.title("📝 AI Essay Checker")
st.write(
    """
Upload your essay as a **PDF or TXT file** and (optionally) describe the assignment.
The app will use an AI model to give you **feedback and improvement suggestions**.
"""
)

# File uploader widget for the essay
uploaded_file = st.file_uploader(
    "Upload your essay file (PDF or TXT)",             # Label above the button
    type=["pdf", "txt"]                                # Only allow pdf / txt files
)

# Text area where the user can describe the assignment / prompt / rubric
assignment_details = st.text_area(
    "Assignment prompt / topic / rubric (optional)",
    help="Example: '5-paragraph persuasive essay about why recycling is important, 2–3 pages.'",
)

# Button that starts the analysis
analyze_button = st.button("Analyze Essay")

# When the user clicks the button, run the analysis
if analyze_button:
    # First, check that the user actually uploaded a file
    if uploaded_file is None:
        st.error("Please upload a PDF or TXT file first.")
    else:
        # Show a spinner while we process the file and call the API
        with st.spinner("Reading your file and generating feedback..."):
            essay_text = extract_text_from_uploaded_file(uploaded_file)

            # If file type is not supported, show an error message
            if essay_text is None:
                st.error(
                    "Sorry, this file type is not supported. "
                    "Please upload a PDF or plain TXT file."
                )
            else:
                # Call our function that talks to OpenAI
                feedback = get_essay_feedback(essay_text, assignment_details)

                # Display the feedback nicely in the app
                st.subheader("Feedback")
                st.write(feedback)

2025-12-08 07:08:29.652 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-08 07:08:29.654 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-08 07:08:29.895 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-12-08 07:08:29.896 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-08 07:08:29.897 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-08 07:08:29.899 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-08 07:08:29.900 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [ ]:
%%writefile app.py
import streamlit as st
from dotenv import load_dotenv
from PyPDF2 import PdfReader
import os
from openai import OpenAI
from google.colab import userdata

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY") or userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

def extract_text_from_pdf(pdf_file):
    reader = PdfReader(pdf_file)
    text_chunks = []
    for page in reader.pages:
        page_text = page.extract_text() or ""
        text_chunks.append(page_text)
    return "\n".join(text_chunks)

def extract_text_from_uploaded_file(uploaded_file):
    if uploaded_file is None:
        return None
    if uploaded_file.type == "application/pdf":
        return extract_text_from_pdf(uploaded_file)
    elif uploaded_file.type == "text/plain":
        bytes_data = uploaded_file.read()
        return bytes_data.decode("utf-8", errors="ignore")
    else:
        return None

def get_essay_feedback(essay_text, assignment_details=None):
    if not essay_text.strip():
        return "I could not detect any text in the uploaded file."

    system_message = (
        "You are a helpful writing tutor. "
        "You carefully read student essays and give specific, constructive feedback. "
        "Focus on structure, clarity, grammar, vocabulary, and whether the essay answers the prompt."
    )

    if assignment_details:
        user_prompt = f"""
Here is an essay a student wrote.

Assignment / prompt or rubric:
{assignment_details}

Essay:
{essay_text}

Please:
1. Briefly summarize what the essay is trying to say.
2. Comment on strengths (organization, ideas, evidence, style).
3. Point out issues (clarity, grammar, word choice, missing parts).
4. Suggest concrete improvements and examples of better sentences.
5. (Optional) Give a rough letter grade (A–F) or score out of 100 and explain why.
"""
    else:
        user_prompt = f"""
Here is a student's essay:

{essay_text}

Please:
1. Briefly summarize what the essay is trying to say.
2. Comment on strengths (organization, ideas, evidence, style).
3. Point out issues (clarity, grammar, word choice, missing parts).
4. Suggest concrete improvements and examples of better sentences.
5. (Optional) Give a rough letter grade (A–F) or score out of 100 and explain why.
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.3,
    )
    return response.choices[0].message.content

st.set_page_config(
    page_title="AI Essay Checker",
    page_icon="📝",
    layout="centered"
)

st.title("📝 AI Essay Checker")
st.write(
    """
Upload your essay as a **PDF or TXT file** and (optionally) describe the assignment.
The app will use an AI model to give you **feedback and improvement suggestions**.
"""
)

uploaded_file = st.file_uploader(
    "Upload your essay file (PDF or TXT)",
    type=["pdf", "txt"]
)

assignment_details = st.text_area(
    "Assignment prompt / topic / rubric (optional)",
    help="Example: '5-paragraph persuasive essay about why recycling is important, 2–3 pages.'",
)

analyze_button = st.button("Analyze Essay")

if analyze_button:
    if uploaded_file is None:
        st.error("Please upload a PDF or TXT file first.")
    else:
        with st.spinner("Reading your file and generating feedback..."):
            essay_text = extract_text_from_uploaded_file(uploaded_file)
            if essay_text is None:
                st.error(
                    "Sorry, this file type is not supported. "
                    "Please upload a PDF or plain TXT file."
                )
            else:
                feedback = get_essay_feedback(essay_text, assignment_details)
                st.subheader("Feedback")
                st.write(feedback)


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.115.62.128:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧your url is: https://yellow-files-cut.loca.lt
curl https://loca.lt/mytunnelpassword
bypass-tunnel-reminder
